# Export .pt → .engine (TensorRT FP16)

Exporta os três modelos ativos usados em `src/main_seg.py` para TensorRT FP16:

| Modelo | Path .pt | `imgsz` | Nota |
|---|---|---|---|
| **Bola** | `models/active/ball_y11m_1280_footar_best.pt` | **960** | Match com `ball_track_imgsz` default |
| **Campo** | `models/active/pitch_v11m_640_footar_best.pt` | **640** | Treinado a 640 |
| **Jogadores (seg)** | `models/active/yolo11m_seg_players.pt` | **1024** | Match com `player_track_imgsz` default |

### ⚠️ Avisos importantes

1. **Hardware-specific**: O `.engine` gerado só funciona nesta GPU (RTX 4060 8GB). Se mudares de máquina, precisas de re-exportar.
2. **Shape estático**: Exportamos com `dynamic=False` → o engine tem `imgsz` fixo. Se no `main_seg.py` usares outro `imgsz`, falha ou fica lento.
3. **FP16 em segmentação**: Da última tentativa, o FP16 do modelo de jogadores degradou a classificação de equipas (máscaras menos precisas contaminavam os histogramas HSV). Se voltar a acontecer, tens duas opções:
   - Passar `retina_masks=True` no `.track()` em runtime (upsample das máscaras à resolução nativa)
   - Re-exportar o modelo de jogadores com `half=False` (FP32)
4. **Tempo de export**: Cada modelo demora ~3–10 minutos. A primeira execução é mais lenta (build do engine).

## 1. Setup

In [1]:
import os
import sys
from pathlib import Path

# Resolver PROJECT_ROOT (duas pastas acima deste notebook)
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
MODELS_DIR = PROJECT_ROOT / 'models' / 'active'

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'MODELS_DIR:   {MODELS_DIR}')
assert MODELS_DIR.exists(), f'Models dir não encontrado: {MODELS_DIR}'

PROJECT_ROOT: C:\_FOOTAR\PD_FOOTAR
MODELS_DIR:   C:\_FOOTAR\PD_FOOTAR\models\active


In [2]:
# Verificar GPU e TensorRT
import torch
print(f'CUDA disponível: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'PyTorch: {torch.__version__}')

try:
    import tensorrt as trt
    print(f'TensorRT: {trt.__version__}')
except ImportError:
    print('❌ tensorrt não instalado. Instala com:')
    print(f'   {sys.executable} -m pip install tensorrt')
    raise

from ultralytics import YOLO
import ultralytics
print(f'Ultralytics: {ultralytics.__version__}')

CUDA disponível: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
PyTorch: 2.5.1+cu121
TensorRT: 10.16.0.72
Ultralytics: 8.4.24


## 2. Configuração dos modelos

**IMPORTANTE**: Os `imgsz` abaixo têm de ser exatamente os mesmos usados em `main_seg.py`. Se mudares os defaults do argparse, re-exporta.

In [3]:
EXPORT_CONFIGS = [
    {
        'name': 'ball',
        'pt_path': MODELS_DIR / 'ball_y11m_1280_footar_best.pt',
        'imgsz': 960,
        'task': 'detect',
    },
    {
        'name': 'pitch',
        'pt_path': MODELS_DIR / 'pitch_v11m_640_footar_best.pt',
        'imgsz': 640,
        'task': 'pose',  # modelo de keypoints
    },
    {
        'name': 'players_seg',
        'pt_path': MODELS_DIR / 'yolo11m_seg_players.pt',
        'imgsz': 1024,
        'task': 'segment',
    },
]

# Parâmetros TensorRT partilhados
EXPORT_KWARGS = dict(
    format='engine',
    half=True,          # FP16
    device=0,           # GPU 0
    simplify=True,      # simplificar ONNX intermédio
    dynamic=False,      # shape estático (mais rápido)
    batch=1,            # inferência single-frame
    workspace=4,        # 4 GB workspace
    verbose=False,
)

# Validar que todos os .pt existem
for cfg in EXPORT_CONFIGS:
    assert cfg['pt_path'].exists(), f'Ficheiro não existe: {cfg["pt_path"]}'
    print(f'✅ {cfg["name"]:15} {cfg["pt_path"].name}  (imgsz={cfg["imgsz"]}, task={cfg["task"]})')

✅ ball            ball_y11m_1280_footar_best.pt  (imgsz=960, task=detect)
✅ pitch           pitch_v11m_640_footar_best.pt  (imgsz=640, task=pose)
✅ players_seg     yolo11m_seg_players.pt  (imgsz=1024, task=segment)


## 3. Exportar Ball detector

Modelo `detect` (YOLO11m a 960px). Export simples, sem caveats.

In [4]:
import time

cfg = EXPORT_CONFIGS[0]  # ball
print(f'📦 Exporting {cfg["name"]}  imgsz={cfg["imgsz"]}')
print(f'   from: {cfg["pt_path"]}')

t0 = time.time()
model = YOLO(str(cfg['pt_path']))
engine_path = model.export(imgsz=cfg['imgsz'], **EXPORT_KWARGS)
dt = time.time() - t0

print(f'\n✅ {cfg["name"]} exported in {dt:.1f}s')
print(f'   engine: {engine_path}')
print(f'   size:   {Path(engine_path).stat().st_size / 1e6:.1f} MB')

📦 Exporting ball  imgsz=960
   from: C:\_FOOTAR\PD_FOOTAR\models\active\ball_y11m_1280_footar_best.pt
Ultralytics 8.4.24  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11m summary (fused): 126 layers, 20,033,116 parameters, 0 gradients, 67.7 GFLOPs

PyTorch: starting from 'C:\_FOOTAR\PD_FOOTAR\models\active\ball_y11m_1280_footar_best.pt' with input shape (1, 3, 960, 960) BCHW and output shape(s) (1, 8, 18900) (38.7 MB)

ONNX: starting export with onnx 1.21.0 opset 17...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success  10.4s, saved as 'C:\_FOOTAR\PD_FOOTAR\models\active\ball_y11m_1280_footar_best.onnx' (76.9 MB)

TensorRT: starting export with TensorRT 10.16.0.72...
TensorRT: input "images" with shape(1, 3, 960, 960) DataType.FLOAT
TensorRT: output "output0" with shape(1, 8, 18900) DataType.FLOAT
TensorRT: building FP16 engine as C:\_FOOTAR\PD_FOOTAR\models\active\ball_y11m_1280_footar_best.engine
TensorRT: export success  355.2s, s

## 4. Exportar Pitch detector

Modelo `pose` (keypoints do campo) a 640px.

In [5]:
cfg = EXPORT_CONFIGS[1]  # pitch
print(f'📦 Exporting {cfg["name"]}  imgsz={cfg["imgsz"]}')
print(f'   from: {cfg["pt_path"]}')

t0 = time.time()
model = YOLO(str(cfg['pt_path']))
engine_path = model.export(imgsz=cfg['imgsz'], **EXPORT_KWARGS)
dt = time.time() - t0

print(f'\n✅ {cfg["name"]} exported in {dt:.1f}s')
print(f'   engine: {engine_path}')
print(f'   size:   {Path(engine_path).stat().st_size / 1e6:.1f} MB')

📦 Exporting pitch  imgsz=640
   from: C:\_FOOTAR\PD_FOOTAR\models\active\pitch_v11m_640_footar_best.pt
Ultralytics 8.4.24  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11m-pose summary (fused): 134 layers, 21,414,067 parameters, 0 gradients, 73.8 GFLOPs

PyTorch: starting from 'C:\_FOOTAR\PD_FOOTAR\models\active\pitch_v11m_640_footar_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 101, 8400) (41.3 MB)

ONNX: starting export with onnx 1.21.0 opset 17...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success  2.5s, saved as 'C:\_FOOTAR\PD_FOOTAR\models\active\pitch_v11m_640_footar_best.onnx' (82.1 MB)

TensorRT: starting export with TensorRT 10.16.0.72...
TensorRT: input "images" with shape(1, 3, 640, 640) DataType.FLOAT
TensorRT: output "output0" with shape(1, 101, 8400) DataType.FLOAT
TensorRT: building FP16 engine as C:\_FOOTAR\PD_FOOTAR\models\active\pitch_v11m_640_footar_best.engine
TensorRT: export success  30

## 5. Exportar Players Segmentation

⚠️ **Atenção**: Este é o export mais sensível. O FP16 pode reduzir a qualidade das máscaras e contaminar o `team_seg.py`. Se em runtime notares classificação de equipas estranha, passa `retina_masks=True` no `.track()` ou re-exporta com `half=False`.

In [6]:
cfg = EXPORT_CONFIGS[2]  # players_seg
print(f'📦 Exporting {cfg["name"]}  imgsz={cfg["imgsz"]}')
print(f'   from: {cfg["pt_path"]}')

t0 = time.time()
model = YOLO(str(cfg['pt_path']))
engine_path = model.export(imgsz=cfg['imgsz'], **EXPORT_KWARGS)
dt = time.time() - t0

print(f'\n✅ {cfg["name"]} exported in {dt:.1f}s')
print(f'   engine: {engine_path}')
print(f'   size:   {Path(engine_path).stat().st_size / 1e6:.1f} MB')

📦 Exporting players_seg  imgsz=1024
   from: C:\_FOOTAR\PD_FOOTAR\models\active\yolo11m_seg_players.pt
Ultralytics 8.4.24  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11m-seg summary (fused): 139 layers, 22,338,396 parameters, 0 gradients, 112.9 GFLOPs

PyTorch: starting from 'C:\_FOOTAR\PD_FOOTAR\models\active\yolo11m_seg_players.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) ((1, 40, 21504), (1, 32, 256, 256)) (43.1 MB)

ONNX: starting export with onnx 1.21.0 opset 17...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success  2.5s, saved as 'C:\_FOOTAR\PD_FOOTAR\models\active\yolo11m_seg_players.onnx' (85.8 MB)

TensorRT: starting export with TensorRT 10.16.0.72...
TensorRT: input "images" with shape(1, 3, 1024, 1024) DataType.FLOAT
TensorRT: output "output0" with shape(1, 40, 21504) DataType.FLOAT
TensorRT: output "output1" with shape(1, 32, 256, 256) DataType.FLOAT
TensorRT: building FP16 engine as C:\_FOOTAR\PD_FO

## 6. Verificar engines com inferência dummy

Carrega cada `.engine` e faz uma predict num tensor aleatório para garantir que o engine abre e corre.

In [7]:
import numpy as np

for cfg in EXPORT_CONFIGS:
    engine_path = cfg['pt_path'].with_suffix('.engine')
    if not engine_path.exists():
        print(f'⚠️  {cfg["name"]}: engine não encontrado em {engine_path}')
        continue

    print(f'🔎 {cfg["name"]}  engine={engine_path.name}')
    try:
        # Task tem de ser passada ao carregar .engine (não é detetada automaticamente)
        model = YOLO(str(engine_path), task=cfg['task'])
        # Frame dummy BGR uint8 no shape correto
        dummy = np.random.randint(0, 255, (cfg['imgsz'], cfg['imgsz'], 3), dtype=np.uint8)
        t0 = time.time()
        # warmup + 5 runs
        _ = model.predict(dummy, imgsz=cfg['imgsz'], verbose=False)
        t1 = time.time()
        for _ in range(5):
            _ = model.predict(dummy, imgsz=cfg['imgsz'], verbose=False)
        dt = (time.time() - t1) / 5
        print(f'   ✅ OK  warmup={t1-t0:.2f}s  avg_inf={dt*1000:.1f}ms  ({1/dt:.0f} FPS)')
    except Exception as e:
        print(f'   ❌ FALHOU: {e}')

🔎 ball  engine=ball_y11m_1280_footar_best.engine
Loading C:\_FOOTAR\PD_FOOTAR\models\active\ball_y11m_1280_footar_best.engine for TensorRT inference...
   ✅ OK  warmup=0.23s  avg_inf=11.4ms  (88 FPS)
🔎 pitch  engine=pitch_v11m_640_footar_best.engine
Loading C:\_FOOTAR\PD_FOOTAR\models\active\pitch_v11m_640_footar_best.engine for TensorRT inference...
   ✅ OK  warmup=0.13s  avg_inf=6.8ms  (147 FPS)
🔎 players_seg  engine=yolo11m_seg_players.engine
Loading C:\_FOOTAR\PD_FOOTAR\models\active\yolo11m_seg_players.engine for TensorRT inference...
   ✅ OK  warmup=0.14s  avg_inf=14.3ms  (70 FPS)


## 7. Como ativar no `main_seg.py`

O `main_seg.py` já tem um branch condicional que carrega `.engine` se a extensão for essa. Basta mudar os paths das constantes:

```python
PLAYER_DETECTION_MODEL_PATH = os.path.join(PROJECT_ROOT, 'models', 'active', 'yolo11m_seg_players.engine')
PITCH_DETECTION_MODEL_PATH  = os.path.join(PROJECT_ROOT, 'models', 'active', 'pitch_v11m_640_footar_best.engine')
BALL_DETECTION_MODEL_PATH   = os.path.join(PROJECT_ROOT, 'models', 'active', 'ball_y11m_1280_footar_best.engine')
```

E confirma que os defaults do argparse batem certo:
- `--player_track_imgsz 1024`
- `--ball_track_imgsz 960`
- Pitch: usa o imgsz do modelo (640)

### Se a classificação de equipas piorar

Adiciona `retina_masks=True` à chamada `player_detection_model.track(...)` em `main_seg.py`. Isso força as máscaras em resolução nativa no post-processing, compensando a perda de precisão do FP16.